# 🛡️ AIRG Enterprise Demo — CyberOps Agent-to-Agent Governance

**Organization 3** · Agent-to-Agent Chain Governance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/othnielObasi/airg-cookbook/blob/main/use_cases/enterprise_demos/org3_cyberops_agent_to_agent.ipynb)

---

This notebook demonstrates **agent-to-agent governance** — where one AI agent delegates tasks to another, and AIRG governs the entire chain.

### Agent Architecture

```
┌──────────────────┐     delegates      ┌──────────────────┐     delegates      ┌──────────────────┐
│  Orchestrator    │ ──────────────────→ │  Analyst Agent   │ ──────────────────→ │  Response Agent  │
│  (triage & plan) │                     │  (investigate)   │                     │  (contain/fix)   │
└──────────────────┘                     └──────────────────┘                     └──────────────────┘
        │                                        │                                        │
        └───────── All calls governed by AIRG ──────────── trace_id links chain ──────────┘
```

Every hop in the chain is evaluated, fingerprinted, and receipted. AIRG detects multi-agent escalation patterns and provides end-to-end trace visibility.

## 1. Setup

In [ ]:
!pip install -q httpx tabulate

import httpx, json, time, os, uuid
from tabulate import tabulate

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOVERNOR_API_KEY")
except Exception:
    API_KEY = os.getenv("GOVERNOR_API_KEY")

if not API_KEY:
    raise RuntimeError("Missing GOVERNOR_API_KEY. Create an API key in your registered AIRG account and add it to Colab Secrets or your environment.")

API_BASE = os.getenv("GOVERNOR_URL", "https://api.airg.nov-tia.com")
HEADERS  = {"X-API-Key": API_KEY, "Content-Type": "application/json"}

ORCHESTRATOR  = "cyberops-orchestrator-v1"
ANALYST       = "cyberops-analyst-v1"
RESPONDER     = "cyberops-responder-v1"

INCIDENT_TRACE = f"incident-{uuid.uuid4().hex[:12]}"
INCIDENT_SESSION = f"incident-{uuid.uuid4().hex[:8]}"

client = httpx.Client(base_url=API_BASE, headers=HEADERS, timeout=30, follow_redirects=True)
print("Connected - CyberOps Org")
print(f"   Orchestrator: {ORCHESTRATOR}")
print(f"   Analyst:      {ANALYST}")
print(f"   Responder:    {RESPONDER}")
print(f"   Incident:     {INCIDENT_TRACE}")


## 2. Configure Governance for CyberOps

In [ ]:
# View current settings
r = client.get("/settings")
if r.status_code == 200:
    s = r.json()
    print("=== CyberOps Governor Settings ===")
    for k in ["kill_switch", "modules_enabled", "fingerprinting_enabled",
              "impact_assessment_enabled", "surge_v2_enabled", "budget_enforcer_enabled",
              "budget_max_evals_per_hour", "budget_max_evals_per_day"]:
        print(f"  {k:40s} = {s.get(k, 'N/A')}")
else:
    print(f"Settings: {r.status_code}")

# Configure for incident response workload
r = client.patch("/settings", json={
    "budget_max_evals_per_hour": 2000,     # Incidents generate bursts
    "budget_max_evals_per_day": 20000,
    "fingerprinting_enabled": True,
    "impact_assessment_enabled": True,
    "verification_required": False,         # Speed over verification in IR
})
if r.status_code == 200:
    print("\n✅ Settings configured for incident response")
else:
    print(f"Update: {r.status_code} — {r.text[:200]}")

In [ ]:
# Module status
r = client.get("/settings/modules")
if r.status_code == 200:
    mods = r.json()
    table = [[m["name"], "✅" if m["enabled"] else "❌", "✅" if m["loaded"] else "❌",
              m["description"][:50]] for m in mods]
    print(tabulate(table, headers=["Module", "On", "Loaded", "Description"], tablefmt="simple"))
else:
    print(f"Modules: {r.status_code}")

## 3. Agent-to-Agent Incident Response

In [ ]:
def gov_eval(tool, args, agent_id, session_id=INCIDENT_SESSION, allowed_tools=None):
    """Evaluate with shared incident trace for cross-agent correlation."""
    payload = {
        "tool": tool,
        "args": args,
        "agent_id": agent_id,
        "session_id": session_id,
        "context": {
            "agent_id": agent_id,
            "session_id": session_id,
            "trace_id": INCIDENT_TRACE,
            "allowed_tools": allowed_tools or []
        }
    }
    r = client.post("/actions/evaluate", json=payload)
    r.raise_for_status()
    d = r.json()
    
    icon = {"allow": "✅", "review": "⚠️", "block": "🚫"}.get(d.get("decision"), "❓")
    print(f"  {icon} [{agent_id[:15]:15s}] {tool:30s} risk={d.get('risk_score','?'):3s}  "
          f"depth={d.get('session_depth',0)}", end="")
    if d.get("chain_pattern"):
        print(f"  🔗 {d['chain_pattern']}", end="")
    if d.get("deviation_count", 0) > 0:
        print(f"  🔍 devs={d['deviation_count']}", end="")
    if d.get("pii_findings_count", 0) > 0:
        print(f"  🛡️ pii={d['pii_findings_count']}", end="")
    if d.get("escalation_severity"):
        print(f"  🚨 {d['escalation_severity']}", end="")
    print()
    return d

### Phase 1: Orchestrator — Triage & Delegation

The orchestrator receives an alert and plans the incident response:

In [ ]:
ORCH_TOOLS = ["triage_alert", "delegate_analysis", "delegate_response",
              "escalate_to_human", "close_incident"]

print("=== Phase 1: Orchestrator Triage ===")
print()

# Step 1: Triage the incoming alert
o1 = gov_eval("triage_alert", {
    "alert_id": "ALERT-2026-0331-001",
    "source": "SIEM",
    "severity": "high",
    "type": "suspicious_login",
    "details": "Multiple failed logins from IP 203.0.113.42 followed by successful auth"
}, ORCHESTRATOR, allowed_tools=ORCH_TOOLS)

time.sleep(0.3)

# Step 2: Delegate to analyst
o2 = gov_eval("delegate_analysis", {
    "alert_id": "ALERT-2026-0331-001",
    "delegate_to": ANALYST,
    "investigation_scope": ["ip_reputation", "user_history", "lateral_movement"],
    "priority": "high"
}, ORCHESTRATOR, allowed_tools=ORCH_TOOLS)

### Phase 2: Analyst — Deep Investigation

The analyst agent investigates the alert with specialized security tools:

In [ ]:
ANALYST_TOOLS = ["query_logs", "check_ip_reputation", "scan_network",
                 "analyze_malware", "query_threat_intel"]

print("=== Phase 2: Analyst Investigation ===")
print()

# Query auth logs
a1 = gov_eval("query_logs", {
    "log_source": "auth_server",
    "query": "source_ip=203.0.113.42 AND event_type=login",
    "time_range": "24h",
    "limit": 1000
}, ANALYST, allowed_tools=ANALYST_TOOLS)

time.sleep(0.3)

# IP reputation check
a2 = gov_eval("check_ip_reputation", {
    "ip": "203.0.113.42",
    "include_geolocation": True,
    "include_threat_feeds": True
}, ANALYST, allowed_tools=ANALYST_TOOLS)

time.sleep(0.3)

# Network scan for lateral movement
a3 = gov_eval("scan_network", {
    "target_subnet": "10.0.1.0/24",
    "scan_type": "connection_map",
    "source_ip": "203.0.113.42",
    "depth": 3
}, ANALYST, allowed_tools=ANALYST_TOOLS)

time.sleep(0.3)

# Threat intel correlation
a4 = gov_eval("query_threat_intel", {
    "indicators": ["203.0.113.42", "suspicious.domain.com"],
    "feeds": ["alienvault", "virustotal", "abuseipdb"]
}, ANALYST, allowed_tools=ANALYST_TOOLS)

### Phase 3: Responder — Containment & Remediation

The response agent takes action to contain the threat:

In [ ]:
RESPONDER_TOOLS = ["block_ip", "isolate_host", "rotate_credentials",
                   "deploy_patch", "restore_backup"]

print("=== Phase 3: Response Actions ===")
print()

# Block the attacking IP
r1 = gov_eval("block_ip", {
    "ip": "203.0.113.42",
    "duration": "permanent",
    "firewall_rule": "BLOCK_INBOUND",
    "reason": "Confirmed brute-force + credential stuffing"
}, RESPONDER, allowed_tools=RESPONDER_TOOLS)

time.sleep(0.3)

# Isolate compromised host
r2 = gov_eval("isolate_host", {
    "hostname": "workstation-jdoe.internal",
    "isolation_level": "network",
    "preserve_forensics": True
}, RESPONDER, allowed_tools=RESPONDER_TOOLS)

time.sleep(0.3)

# Rotate compromised credentials
r3 = gov_eval("rotate_credentials", {
    "user": "jdoe",
    "credential_types": ["password", "api_keys", "ssh_keys"],
    "notify_user": True,
    "force_mfa_re_enrollment": True
}, RESPONDER, allowed_tools=RESPONDER_TOOLS)

time.sleep(0.3)

# Dangerous: attempt to wipe logs (should be blocked)
r4 = gov_eval("execute_command", {
    "command": "rm -rf /var/log/auth.log",
    "target": "workstation-jdoe.internal",
    "reason": "Clean up traces"  # Suspicious!
}, RESPONDER, allowed_tools=RESPONDER_TOOLS)

### Phase 4: Orchestrator — Close Incident

In [ ]:
print("=== Phase 4: Close Incident ===")
print()

o3 = gov_eval("close_incident", {
    "alert_id": "ALERT-2026-0331-001",
    "resolution": "contained",
    "actions_taken": [
        "Blocked attacking IP 203.0.113.42",
        "Isolated workstation-jdoe.internal",
        "Rotated jdoe credentials + forced MFA re-enrollment",
        "Blocked unauthorized log deletion attempt"
    ],
    "analysts": [ANALYST],
    "responders": [RESPONDER]
}, ORCHESTRATOR, allowed_tools=ORCH_TOOLS)

## 4. Output Scan — Protect Incident Data

In [ ]:
scan_r = client.post("/actions/scan-output", json={
    "text": "Incident summary: User jdoe (SSN: 987-65-4321) account compromised via "
             "credential stuffing from 203.0.113.42. AWS access key AKIA1234567890ABCDEF "
             "and secret key wJalrXUtnFEMI/K7MDENG/bPxRfiCYEXAMPLEKEY were exposed. "
             "See https://evil.com/exfil?data=leaked_creds for IOC details.",
    "agent_id": ORCHESTRATOR
})
if scan_r.status_code == 200:
    s = scan_r.json()
    print(f"Scan: safe={s.get('safe')}  risk={s.get('risk_score', 0)}")
    for f in s.get("findings", []):
        icon = "❌" if f.get("result") == "fail" else "⚠️"
        print(f"  {icon} [{f.get('check','?')}] {f.get('detail','')[:80]}  risk+{f.get('risk_contribution',0)}")
else:
    print(f"Scan: {scan_r.status_code} — {scan_r.text[:200]}")

## 5. Verification — Validate Response Actions

In [ ]:
# Verify the IP block action
action_id = r1.get("action_id")
if action_id:
    v_r = client.post("/actions/verify", json={
        "action_id": action_id,
        "tool": "block_ip",
        "result": {
            "status": "applied",
            "firewall_rule_id": "FW-RULE-42",
            "effective_at": "2026-03-31T15:00:00Z",
            "confirmed_blocked": True,
            "test_connection_refused": True
        },
        "context": {"agent_id": RESPONDER, "session_id": INCIDENT_SESSION,
                     "trace_id": INCIDENT_TRACE}
    })
    if v_r.status_code == 200:
        v = v_r.json()
        print(f"Verification: {v.get('verification')}  |  Risk delta: {v.get('risk_delta',0):+d}")
        if v.get("findings"):
            for f in v["findings"]:
                icon = "✅" if f["result"] == "pass" else "❌"
                print(f"  {icon} {f['check']:30s} risk+{f.get('risk_contribution',0)}")
    else:
        print(f"Verify: {v_r.status_code} — {v_r.text[:200]}")

## 6. Cross-Agent Impact & SURGE

In [ ]:
# Per-agent impact
for agent in [ORCHESTRATOR, ANALYST, RESPONDER]:
    print(f"\n--- Impact: {agent} ---")
    r = client.get("/impact/assess", params={"agent_id": agent, "window_hours": 24})
    if r.status_code == 200:
        imp = r.json()
        print(f"  Evaluations: {imp.get('total_evaluations', 'N/A')}")
        print(f"  Avg risk:    {imp.get('avg_risk_score', 'N/A')}")
        print(f"  Block rate:  {imp.get('block_rate', 'N/A')}")
        if imp.get("chain_patterns"):
            print(f"  Chains:      {imp['chain_patterns']}")
    else:
        print(f"  {r.status_code}: {r.text[:100]}")

In [ ]:
# SURGE receipts — each agent generates its own receipt chain
print("=== SURGE v2 Receipts ===")
r = client.get("/surge/v2/receipts", params={"limit": 10})
if r.status_code == 200:
    receipts = r.json()
    if isinstance(receipts, list) and receipts:
        for rcpt in receipts:
            print(f"  #{rcpt.get('sequence','?'):4d}  digest={str(rcpt.get('digest',''))[:30]}...")
    else:
        print("  No receipts yet")
else:
    print(f"  {r.status_code}")

print("\n=== SURGE v2 Chain Status ===")
r = client.get("/surge/v2/status")
if r.status_code == 200:
    s = r.json()
    for k, v in s.items():
        if not isinstance(v, (dict, list)):
            print(f"  {k}: {v}")
    if s.get("pricing"):
        print(f"  pricing: {json.dumps(s['pricing'])}")
else:
    print(f"  {r.status_code}")

## 7. Cross-Agent Trace Visibility

All three agents share the same `trace_id`, creating a unified incident timeline:

In [ ]:
# Get all spans for the incident trace
print(f"=== Incident Trace: {INCIDENT_TRACE} ===")

def response_items(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        return raw.get("items") or raw.get("traces") or raw.get("spans") or raw.get("data") or []
    return []

r = client.get("/traces", params={"trace_id": INCIDENT_TRACE, "limit": 20})
if r.status_code == 200:
    spans = response_items(r.json())
    if spans:
        table = []
        for s in spans:
            table.append([
                (s.get("name") or s.get("span_name") or "?")[:35],
                s.get("kind", "?"),
                s.get("status", "?"),
                f"{s.get('duration_ms', '?')}ms",
                (s.get("agent_id") or "?")[:18]
            ])
        print(tabulate(table, headers=["Span", "Kind", "Status", "Duration", "Agent"], tablefmt="simple"))
    else:
        print("  No trace spans yet")
else:
    print(f"  Traces: {r.status_code} - {r.text[:200]}")


## 8. Combined Audit Log

In [ ]:
print("=== Full Incident Audit Log ===")

def response_items(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        return raw.get("items") or raw.get("actions") or raw.get("data") or []
    return []

r = client.get("/actions", params={"limit": 20})
if r.status_code == 200:
    actions = response_items(r.json())
    table = []
    for a in actions:
        table.append([
            a.get("id"),
            (a.get("agent_id") or "?")[:18],
            a.get("tool","?")[:22],
            a.get("decision","?"),
            a.get("risk_score","?"),
            a.get("chain_pattern") or "-",
        ])
    print(tabulate(table, headers=["ID","Agent","Tool","Decision","Risk","Chain"], tablefmt="grid"))
else:
    print(f"Actions: {r.status_code} - {r.text[:200]}")


## 9. Summary — Agent-to-Agent Governance

This demo demonstrated **agent-to-agent chain governance** across 3 CyberOps agents:

```
Orchestrator → Analyst → Responder
   (triage)    (investigate)  (contain)
```

| Capability | Result |
|-----------|--------|
| Multi-agent tracing | ✅ Shared `trace_id` across all 3 agents |
| Independent fingerprinting | ✅ Per-agent behavioural baselines |
| Chain detection | ✅ Multi-step pattern recognition across agents |
| Cross-agent impact | ✅ Org-wide risk visibility |
| Blocked rogue actions | ✅ Log deletion attempt blocked |
| SURGE receipt chain | ✅ All evaluations cryptographically attested |
| Settings configuration | ✅ Org-wide IR-optimized settings |
| Output scanning | ✅ Credential/PII leak prevention |
| Verification | ✅ Post-action compliance validation |

AIRG provides **zero-trust governance** for agent-to-agent workflows: every delegation, investigation, and response action is evaluated, fingerprinted, and receipted.

In [ ]:
client.close()
print("✅ CyberOps agent-to-agent demo complete.")
print(f"   Incident trace: {INCIDENT_TRACE}")
print(f"   All actions available via GET /actions for compliance audit")